# v5 LoRA: 纯风格迁移
**attention-only, top 6 layers, rank=8**

Colab T4 (16G) · Qwen2.5-7B · ~1.2M 可训参数 · 预计 40-60min

## 0. 前置准备
将 `train.json` 上传到 Google Drive 根目录（或修改下方 `DATA_PATH`）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes

In [ ]:
import json
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import Dataset

DATA_PATH = "/content/drive/MyDrive/train.json"
OUTPUT_DIR = "/content/drive/MyDrive/chat_style_lora_v5"
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. 加载数据

In [ ]:
with open(DATA_PATH, encoding="utf-8") as f:
    raw = json.load(f)
print(f"{len(raw)} samples")

# Preview
for m in raw[0]["conversations"][:3]:
    print(f"  [{m['from']}] {m['value'][:60]}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_sample(conv):
    messages = []
    for m in conv["conversations"]:
        role = "user" if m["from"] == "human" else ("assistant" if m["from"] == "gpt" else "system")
        messages.append({"role": role, "content": m["value"]})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

formatted = [format_sample(s) for s in raw]
print(formatted[0][:200])

In [ ]:
dataset = Dataset.from_dict({"text": formatted})

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=2048, padding=False)

dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(f"Tokenized: {len(dataset)} samples")

## 2. 加载模型 + v5 LoRA

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()  # 省显存

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        r"model\.layers\.2[2-7]\.self_attn\.q_proj",
        r"model\.layers\.2[2-7]\.self_attn\.k_proj",
        r"model\.layers\.2[2-7]\.self_attn\.v_proj",
        r"model\.layers\.2[2-7]\.self_attn\.o_proj",
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. 训练

In [ ]:
training_args = TrainingArguments(
    output_dir="/tmp/lora_v5_checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=500,
    bf16=True,
    overwrite_output_dir=True,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

## 4. 保存到 Google Drive

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f" Done → {OUTPUT_DIR}")

In [ ]:
!ls -lh {OUTPUT_DIR}/adapter_model.safetensors

## 5. 快速测试

In [ ]:
test_prompts = ["想你了", "今天好累", "在干嘛呢", "吃过了吗"]
for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "你是个爱撒娇的女孩，正在和男朋友聊天"},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True).cuda()
    outputs = model.generate(inputs, max_new_tokens=64, temperature=0.7, do_sample=True)
    reply = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    print(f"Q: {prompt}  →  {reply}")